In [2]:
import os

print("📍 Hiện tại Python đang đứng ở thư mục:", os.getcwd())
print("📂 Danh sách các file mà Python ĐANG NHÌN THẤY tại đây gồm có:")
for f in os.listdir('.'):
    print("   -", f)

📍 Hiện tại Python đang đứng ở thư mục: f:\Workspace\PYTHON\Projects\FeedbackAnalysis\notebooks
📂 Danh sách các file mà Python ĐANG NHÌN THẤY tại đây gồm có:
   - acronyms_dictionary.txt
   - baseline_model.ipynb
   - main_phobert.ipynb
   - vncorenlp


In [3]:
# =====================================================================
# CELL 1: KHỞI TẠO VNCORENLP
# =====================================================================
import os
import py_vncorenlp

# Trỏ đường dẫn tuyệt đối thẳng vào phòng 'vncorenlp'
save_dir = os.path.abspath('./vncorenlp')

#Kiểm tra xem trong RAM đã bật VnCoreNLP chưa, nếu có rồi thì bỏ qua không bật lại
if 'rdrsegmenter' not in globals():
    print("⏳ Lần đầu chạy: Đang kích hoạt bộ phân đoạn từ ghép VnCoreNLP...")
    rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=save_dir)
    print("✅ KÍCH HOẠT THÀNH CÔNG: Bộ máy VnCoreNLP đã sẵn sàng!")
else:
    print("🔄 Hệ thống nhận diện bộ máy VnCoreNLP đã được bật sẵn trong RAM từ trước.")

⏳ Lần đầu chạy: Đang kích hoạt bộ phân đoạn từ ghép VnCoreNLP...
✅ KÍCH HOẠT THÀNH CÔNG: Bộ máy VnCoreNLP đã sẵn sàng!


In [4]:
import os
import re
from underthesea import text_normalize

# =====================================================================
# CELL 2: XÂY DỰNG HỆ THỐNG TIỀN XỬ LÝ DỮ LIỆU (DATA PREPROCESSING PIPELINE)
# Mục tiêu: Tích hợp chuẩn hóa Unicode, khôi phục từ viết tắt và phân đoạn từ (Word Segmentation)
# =====================================================================

# 1. Khởi tạo và nạp từ điển viết tắt/từ lóng chuyên ngành
dict_path = "acronyms_dictionary.txt"
if not os.path.exists(dict_path):
    # Dự phòng trường hợp hệ thống đọc sai thư mục làm việc (Working Directory)
    dict_path = "../acronyms_dictionary.txt" 

def load_acronyms(file_path):
    """
    Hàm đọc tệp cấu hình từ điển và chuyển đổi thành cấu trúc Cấu trúc dữ liệu Dictionary.
    Bỏ qua các dòng trống và dòng chú thích (bắt đầu bằng ký tự #).
    """
    acronyms_dict = {}
    if not os.path.exists(file_path):
        print(f"[CẢNH BÁO] Không tìm thấy tệp từ điển tại: {file_path}")
        return acronyms_dict
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): 
                continue
            if '=' in line:
                key, value = line.split('=', 1)
                acronyms_dict[key.strip().lower()] = value.strip().lower()
    return acronyms_dict

ACRONYMS_DICT = load_acronyms(dict_path)
print(f"[THÔNG TIN] Đã nạp thành công bộ từ điển. Tổng số lượng từ khóa: {len(ACRONYMS_DICT)}")

# 2. Xây dựng hàm tiền xử lý văn bản đa tầng
def clean_sentence_pipeline(text):
    """
    Hàm thực hiện tiền xử lý văn bản qua 3 giai đoạn độc lập:
    - Giai đoạn 1: Chuẩn hóa bộ gõ và Unicode tiếng Việt (Sử dụng thư viện Underthesea).
    - Giai đoạn 2: Khôi phục từ viết tắt dựa trên từ điển chuyên ngành bằng biểu thức chính quy (Regex).
    - Giai đoạn 3: Phân đoạn từ ghép tiếng Việt (Word Segmentation) bằng mô hình VnCoreNLP.
    """
    # Xử lý ngoại lệ với dữ liệu rỗng hoặc không hợp lệ
    if not isinstance(text, str) or text.strip() == "":
        return ""
    
    text = text.lower()
    
    # GIAI ĐOẠN 1: Chuẩn hóa Unicode tiếng Việt (ví dụ: oà -> hòa) và loại bỏ khoảng trắng thừa
    text = text_normalize(text)
    
    # GIAI ĐOẠN 2: Khôi phục từ viết tắt chuyên ngành
    for shortcut, full_word in ACRONYMS_DICT.items():
        # Định dạng quy tắc: Nếu từ viết tắt là văn bản thông thường (Alphanumeric)
        # Sử dụng ranh giới từ (\b) để tránh thay thế lỗi khi từ viết tắt nằm trong một từ khác
        if re.match(r'^\w+$', shortcut, flags=re.UNICODE):
            text = re.sub(r'\b' + shortcut + r'\b', full_word, text)
        # Định dạng quy tắc: Nếu là ký tự đặc biệt (Ví dụ: emoji <3, @@), tiến hành thay thế trực tiếp
        else:
            text = text.replace(shortcut, full_word)
            
    # GIAI ĐOẠN 3: Phân đoạn từ ghép (Word Segmentation)
    try:
        # Biến rdrsegmenter đã được kế thừa từ quá trình khởi tạo ở Cell 2
        sentences = rdrsegmenter.word_segment(text)
        # Trích xuất câu đã được gạch nối (mặc định lấy câu đầu tiên do cấu trúc dữ liệu trả về là mảng)
        text_segmented = sentences[0] if sentences else text
    except Exception as e:
        # Cơ chế dự phòng: Trả về văn bản của Giai đoạn 2 nếu VnCoreNLP gặp lỗi hệ thống
        text_segmented = text
        
    return text_segmented

# =====================================================================
# Kiểm thử đơn vị (Unit Testing) để đánh giá hệ thống
# =====================================================================
test_str = "    hôm nay   sv   đi học hoà bình ,   ko có gì  xảy ra  <3 "
print("\n[KIỂM TRA CHẤT LƯỢNG TIỀN XỬ LÝ]")
print(f" - Văn bản gốc : '{test_str}'")
print(f" - Sau xử lý   : '{clean_sentence_pipeline(test_str)}'")

[THÔNG TIN] Đã nạp thành công bộ từ điển. Tổng số lượng từ khóa: 579

[KIỂM TRA CHẤT LƯỢNG TIỀN XỬ LÝ]
 - Văn bản gốc : '    hôm nay   sv   đi học hoà bình ,   ko có gì  xảy ra  <3 '
 - Sau xử lý   : 'hôm nay sinh viên đi học hòa bình , không có gì xảy ra colonlove'


In [5]:
# =====================================================================
# CELL 3: TIẾN HÀNH DỌN SẠCH Train CÂU VÀ XUẤT FILE train_cleand_PhoBERT.csv
# =====================================================================
import os
import pandas as pd

# 1. Trỏ đường dẫn an toàn đến kho dữ liệu (chống vụ lạc đường giống Cell 2)
base_data_path = "../data"
if not os.path.exists(base_data_path):
    base_data_path = "../../data"  # Lùi thêm 1 bước nếu Python đang bị kẹt ở phòng vncorenlp

raw_train_path = os.path.join(base_data_path, "raw", "train")
processed_path = os.path.join(base_data_path, "processed")

print(f"📍 Đang định vị kho dữ liệu tại: {raw_train_path}")

# 2. Bốc 3 file txt lên
try:
    with open(os.path.join(raw_train_path, 'sents.txt'), 'r', encoding='utf-8') as f:
        raw_texts = [line.strip() for line in f.readlines()]
    with open(os.path.join(raw_train_path, 'sentiments.txt'), 'r', encoding='utf-8') as f:
        sentiments = [int(line.strip()) for line in f.readlines()]
    with open(os.path.join(raw_train_path, 'topics.txt'), 'r', encoding='utf-8') as f:
        topics = [int(line.strip()) for line in f.readlines()]
        
    df_raw = pd.DataFrame({'raw_text': raw_texts, 'sentiment': sentiments, 'topic': topics})
    print(f"✅ Đã nạp thành công {len(df_raw)} câu đánh giá gốc!")
    
except FileNotFoundError as e:
    print(f"❌ Lỗi: Không tìm thấy file dữ liệu gốc! Chi tiết: {e}")

# 3. Chạy dây chuyền gọt rửa cho toàn bộ dữ liệu
print("\n⏳ Hệ thống đang tiến hành xử lý hàng loạt 11.425 câu (Vui lòng đợi 1-2 phút)...")
# Hàm apply sẽ tự động nhét từng câu vào cái clean_sentence_pipeline ở Cell 2
df_raw['clean_text_PhoBERT'] = df_raw['raw_text'].apply(clean_sentence_pipeline)

# Lọc bỏ các câu rỗng (nếu có để tránh lỗi lúc train)
df_raw = df_raw[df_raw['clean_text_PhoBERT'] != ""]

# 4. Xuất xưởng file CSV
output_file = os.path.join(processed_path, "train_clean_PhoBERT.csv")
df_raw[['raw_text', 'clean_text_PhoBERT', 'sentiment', 'topic']].to_csv(output_file, index=False, encoding='utf-8')

print("-" * 60)
print("👀 NGHIỆM THU DÒNG THỨ 2 CỦA FILE MỚI:")
print(f"  - Câu gốc: {df_raw['raw_text'].iloc[1]}")
print(f"  - Câu sạch: {df_raw['clean_text_PhoBERT'].iloc[1]}")
print("-" * 60)
print(f"🎉 Đã đóng gói thành công file: {output_file}")
print(f"Tổng số mẫu chuẩn chỉnh đã sẵn sàng huấn luyện: {len(df_raw)} mẫu.")

📍 Đang định vị kho dữ liệu tại: ../data\raw\train
✅ Đã nạp thành công 11426 câu đánh giá gốc!

⏳ Hệ thống đang tiến hành xử lý hàng loạt 11.425 câu (Vui lòng đợi 1-2 phút)...
------------------------------------------------------------
👀 NGHIỆM THU DÒNG THỨ 2 CỦA FILE MỚI:
  - Câu gốc: nhiệt tình giảng dạy , gần gũi với sinh viên .
  - Câu sạch: nhiệt tình giảng dạy , gần gũi với sinh viên .
------------------------------------------------------------
🎉 Đã đóng gói thành công file: ../data\processed\train_clean_PhoBERT.csv
Tổng số mẫu chuẩn chỉnh đã sẵn sàng huấn luyện: 11426 mẫu.


In [6]:
# =====================================================================
# CELL 4: TOKENIZATION - MÃ HÓA VĂN BẢN SANG ĐỊNH DẠNG VECTOR SỐ
# Mục tiêu: Chuyển đổi ngôn ngữ tự nhiên thành ma trận số (Tensors) để nạp vào PhoBERT
# =====================================================================
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# 1. Định vị đường dẫn an toàn đến tệp dữ liệu đã làm sạch
# Cơ chế dự phòng trường hợp thư mục làm việc hiện tại (CWD) bị thay đổi bởi tiến trình VnCoreNLP
data_path = "../data/processed/train_clean_PhoBERT.csv"
if not os.path.exists(data_path):
    data_path = "../../data/processed/train_clean_PhoBERT.csv"

print(f"[THÔNG TIN] Đường dẫn tệp dữ liệu được xác định tại: {os.path.abspath(data_path)}")

# 2. Đọc tập dữ liệu đã tiền xử lý chuyên ngành
try:
    df = pd.read_csv(data_path)
    # Loại bỏ các hàng có giá trị rỗng ở các cột quan trọng nhằm đảm bảo tính toàn vẹn dữ liệu
    df = df.dropna(subset=['clean_text_PhoBERT', 'sentiment'])
    print(f"✅ Nạp thành công tập dữ liệu sạch. Tổng số lượng mẫu: {len(df)}")
except FileNotFoundError as e:
    raise FileNotFoundError(f"[LỖI] Hệ thống không thể truy cập tệp dữ liệu tại đường dẫn được chỉ định. Chi tiết: {e}")

# 3. Khởi tạo bộ mã hóa từ vựng (Tokenizer) mã nguồn mở của mô hình PhoBERT
print("\n⏳ Đang kết nối và tải cấu hình Tokenizer PhoBERT (phiên bản vinai/phobert-base-v2)...")
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
print("✅ Tải cấu hình Tokenizer thành công!")

# 4. Phân tách tập dữ liệu theo tỷ lệ 80% Huấn luyện (Train) và 20% Kiểm thử/Đánh giá (Validation)
X = df['clean_text_PhoBERT'].tolist()
y = df['sentiment'].tolist()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"📊 Cấu trúc phân chia dữ liệu huấn luyện:")
print(f"  - Tập Huấn luyện (Train Dataset) : {len(X_train)} mẫu")
print(f"  - Tập Đánh giá (Val Dataset)     : {len(X_val)} mẫu")

# 5. Thực hiện mã hóa hàng loạt văn bản văn bản sang cấu trúc ma trận kỹ thuật số PyTorch (Tensors)
print("\n⏳ Đang thực hiện quá trình Tokenization và Padding đồng bộ hình khối...")
# Quy định các tham số chuẩn hóa cấu trúc vector đầu vào:
# - truncation=True: Ngắt bỏ các ký tự vượt quá giới hạn độ dài tối đa quy định
# - padding=True: Chèn thêm token đệm (mã số 0) đối với các câu ngắn để đồng bộ kích thước ma trận
# - max_length=128: Giới hạn độ dài chuỗi mã hóa tối đa là 128 tokens
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128, return_tensors="pt")
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=128, return_tensors="pt")

# =====================================================================
# Kiểm tra cấu trúc dữ liệu đầu ra sau khi số hóa (Sanity Check)
# =====================================================================
print("-" * 60)
print("[KIỂM TRA ĐẦU RA KỸ THUẬT SỐ - MẪU ĐẦU TIÊN TẬP TRAIN]")
print(f" - Văn bản ngôn ngữ tự nhiên : {X_train[0]}")
# Trích xuất 25 số định danh đầu tiên (Token IDs) để nghiệm thu trực quan
print(f" - Định dạng Vector số (AI)  : {train_encodings['input_ids'][0][:25].tolist()}...")
print("-" * 60)
print("🎉 HOÀN THÀNH: Toàn bộ văn bản đã được mã hóa thành cấu trúc Tensors chuẩn PyTorch, sẵn sàng cấu hình Model!")

[THÔNG TIN] Đường dẫn tệp dữ liệu được xác định tại: f:\Workspace\PYTHON\Projects\FeedbackAnalysis\data\processed\train_clean_PhoBERT.csv
✅ Nạp thành công tập dữ liệu sạch. Tổng số lượng mẫu: 11426

⏳ Đang kết nối và tải cấu hình Tokenizer PhoBERT (phiên bản vinai/phobert-base-v2)...


✅ Tải cấu hình Tokenizer thành công!
📊 Cấu trúc phân chia dữ liệu huấn luyện:
  - Tập Huấn luyện (Train Dataset) : 9140 mẫu
  - Tập Đánh giá (Val Dataset)     : 2286 mẫu

⏳ Đang thực hiện quá trình Tokenization và Padding đồng bộ hình khối...
------------------------------------------------------------
[KIỂM TRA ĐẦU RA KỸ THUẬT SỐ - MẪU ĐẦU TIÊN TẬP TRAIN]
 - Văn bản ngôn ngữ tự nhiên : nội dụng môn học không liên quan đến đề thi .
 - Định dạng Vector số (AI)  : [0, 2151, 8410, 1002, 222, 17, 2657, 2665, 30, 1294, 201, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]...
------------------------------------------------------------
🎉 HOÀN THÀNH: Toàn bộ văn bản đã được mã hóa thành cấu trúc Tensors chuẩn PyTorch, sẵn sàng cấu hình Model!


In [7]:
# =====================================================================
# CELL 5: CẤU HÌNH KIẾN TRÚC MẠNG NƠ-RON VÀ BỘ HUẤN LUYỆN (TRAINER)
# Mục tiêu: Đóng gói dữ liệu PyTorch Dataset, khởi tạo PhoBERT Sequence Classification
# =====================================================================
import os
# 🚨 BÍ KÍP VƯỢT TƯỜNG LỬA: Ép thư viện tải từ trạm trung chuyển thay vì trang chủ
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# 1. Kế thừa và định nghĩa lớp Dataset chuẩn PyTorch cho bài toán NLP
class FeedbackDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FeedbackDataset(train_encodings, y_train)
val_dataset = FeedbackDataset(val_encodings, y_val)

# 2. Phân tích số lượng nhãn phân loại từ dữ liệu gốc
unique_labels = set(y_train)
num_labels = len(unique_labels)
print(f"📊 [PHÂN TÍCH DỮ LIỆU] Bài toán phân loại {num_labels} nhãn (Classes): {unique_labels}")

# 3. Tải kiến trúc mạng nơ-ron PhoBERT qua Trạm trung chuyển
print("\n⏳ Đang khởi tạo kiến trúc mạng nơ-ron PhoBERT (AutoModelForSequenceClassification)...")
print("📡 Đang kết nối qua máy chủ dự phòng hf-mirror.com để tránh nghẽn mạng...")
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base-v2", num_labels=num_labels)
print("✅ Tải kiến trúc mô hình thành công!")

# 4. Hàm tính toán độ đo đánh giá mô hình
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

# 5. Khai báo Siêu tham số Huấn luyện (Hyperparameters)
training_args = TrainingArguments(
    output_dir='./phobert_results',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",      # Dùng eval_strategy thay vì evaluation_strategy cho bản mới
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

# 6. Khởi tạo hệ thống Huấn luyện tự động (Trainer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("-" * 60)
print("🎉 THIẾT LẬP HOÀN TẤT! Hệ thống Trainer đã sẵn sàng.")

📊 [PHÂN TÍCH DỮ LIỆU] Bài toán phân loại 3 nhãn (Classes): {0, 1, 2}

⏳ Đang khởi tạo kiến trúc mạng nơ-ron PhoBERT (AutoModelForSequenceClassification)...
📡 Đang kết nối qua máy chủ dự phòng hf-mirror.com để tránh nghẽn mạng...


f:\Workspace\PYTHON\Projects\FeedbackAnalysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--vinai--phobert-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5502.72it/s]
[transformers] RobertaForS

✅ Tải kiến trúc mô hình thành công!
------------------------------------------------------------
🎉 THIẾT LẬP HOÀN TẤT! Hệ thống Trainer đã sẵn sàng.


In [8]:
# =====================================================================
# CELL 6: KÍCH HOẠT HUẤN LUYỆN VÀ LƯU MÔ HÌNH (FINE-TUNING)
# =====================================================================

print("🔥 BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN (Vui lòng không tắt máy)...")
print("👉 Mẹo: Nếu máy bạn có GPU, tốc độ sẽ nhanh hơn gấp 10 lần.")

# 1. Kích hoạt Trainer để bắt đầu cày dữ liệu
train_result = trainer.train()

# 2. In báo cáo tóm tắt quá trình học
print("\n📊 BÁO CÁO KẾT QUẢ HUẤN LUYỆN:")
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

# 3. Ép hệ thống lưu lại bộ não (Trọng số) xịn nhất vào ổ cứng
output_dir = "./phobert_final_model"
print(f"\n💾 Đang xuất xưởng mô hình và từ điển ra thư mục: {output_dir}")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("🎉 XUẤT SẮC! Đã huấn luyện và lưu trữ mô hình thành công.")

🔥 BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN (Vui lòng không tắt máy)...
👉 Mẹo: Nếu máy bạn có GPU, tốc độ sẽ nhanh hơn gấp 10 lần.


f:\Workspace\PYTHON\Projects\FeedbackAnalysis\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 